<a href="https://colab.research.google.com/github/dildar-ai/ML-pipeline-Flyrank-Assignment-1-/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN') # Retrieve HF token securely from Colab secrets
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Optional: Verify connection and data access
# con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')").show()

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dildar-ai/ML-pipeline-Flyrank-Assignment-1-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

One row in this dataset represents the daily performance of a single content item. The primary keys are `content_hash_id` and `report_date`. The time window for analysis and feature engineering will focus on a mid-panel month, such as `2026-03`, to ensure the data is representative and to avoid leakage from the designated test month (June 2026).

In [13]:
# Assuming 'rel' and 'con' are already defined from previous cells (cell `C-4gweZqFhg4`).
# For verification, we'll focus on a mid-panel month, e.g., 2026-03.
target_month = '2026-03'
table_path = f"{rel}/fact_content_daily_performance/month={target_month}/*.parquet"

print(f"Verifying unit of analysis and time window for month: {target_month}")

# Query to get a sample of the data and show relevant columns
df_sample = con.sql(f"""
    SELECT
        content_hash_id,    -- Corrected from content_id
        report_date,        -- Corrected from date
        gsc_impressions,    -- Corrected from impressions
        gsc_clicks          -- Assuming gsc_clicks based on gsc_impressions
    FROM
        read_parquet('{table_path}')
    LIMIT 10
""").fetchdf()

print(f"Sample data from '{table_path}':")
display(df_sample)

# Verify the grain: unique combinations of content_hash_id and report_date
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_content_ids,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT (content_hash_id, report_date)) AS distinct_content_date_combinations
    FROM
        read_parquet('{table_path}')
""").fetchdf()

print(f"\nGrain verification for month '{target_month}':")
display(grain_check)

# Verify the time window
time_window_check = con.sql(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM
        read_parquet('{table_path}')
""").fetchdf()

print(f"\nTime window verification for month '{target_month}':")
display(time_window_check)

Verifying unit of analysis and time window for month: 2026-03
Sample data from 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet':


,content_hash_id,report_date,gsc_impressions,gsc_clicks
0,content_b7e512995f79d5a6,2026-03-01,20,0
1,content_05597932fe4da067,2026-03-01,1,0
2,content_7a105f548d9c6916,2026-03-01,125,1
3,content_905aa32a0230694e,2026-03-01,7,0
4,content_a3ea9792f793ec72,2026-03-01,11,0
5,content_36c36abc7650d7af,2026-03-01,239,1
6,content_a7da352b73b02668,2026-03-01,191,0
7,content_05434271b257bb68,2026-03-01,55,0
8,content_d056587ff7faca0c,2026-03-01,77,0
9,content_bfd1e41c2af250c8,2026-03-01,2,0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Grain verification for month '2026-03':


,total_rows,distinct_content_ids,distinct_dates,distinct_content_date_combinations
0,9841378,331437,31,9841378



Time window verification for month '2026-03':


,min_date,max_date
0,2026-03-01,2026-03-31


In [15]:
# Temporary schema inspection cell removed.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Unit of Analysis:** Daily performance of a single content item, identified by `content_hash_id` and `report_date`.

**Features (knowable at prediction time):**
*   `gsc_impressions`: Number of times the content was shown (from Google Search Console) on a given day.
*   `gsc_clicks`: Number of times the content was clicked (from Google Search Console) on a given day.
*   `gsc_ctr_7d_ma`: The 7-day moving average of Click-Through Rate (CTR) for the content item (derived from `gsc_clicks` and `gsc_impressions`), representing its recent performance trend. (This will be a *derived* feature).

**Label (what we predict or rank):**
*   `gsc_clicks_next_day`: The number of Google Search Console clicks a content item receives on the day immediately following the feature observation date. This serves as a proxy for future engagement.

**Context (identifiers and metadata):**
*   `content_hash_id`: Unique identifier for the content item.
*   `report_date`: The observation date for the features.

**Excluded (and why):**
*   Any future-dated metrics or aggregated metrics that include the label's target day or later (`gsc_clicks` on the *next* day, `gsc_impressions` on the *next* day). Including these as features would lead to data leakage, as they would not be available at the time of prediction for the label. Specifically, `gsc_clicks_next_day` *itself* would be excluded as a feature (but is used as the label), as it is the future outcome we are trying to predict. Any derived metrics based on `gsc_clicks_next_day` would also be excluded from features to prevent leakage.

In [14]:
import pandas as pd

target_month = '2026-03'
table_path = f"{rel}/fact_content_daily_performance/month={target_month}/*.parquet"

print(f"Verifying fields for month: {target_month}")

# Query to select features, context, and a mock label (for demonstration)
# Note: actual 'gsc_clicks_next_day' requires a self-join or window function for a real label.
# Here we just show the relevant columns that would be used.
df_fields_sample = con.sql(f"""
    SELECT
        content_hash_id,         -- Context (corrected from content_id)
        report_date,             -- Context (corrected from date)
        gsc_impressions,        -- Feature (corrected from impressions)
        gsc_clicks             -- Feature (assuming gsc_clicks)
    FROM
        read_parquet('{table_path}')
    ORDER BY content_hash_id, report_date
    LIMIT 10
""").fetchdf()

print(f"Sample of identified fields from '{table_path}':")
display(df_fields_sample)

# To demonstrate the 'gsc_clicks_next_day' label, we need a self-join or window function.
# This query simulates how the label would be constructed (for a small sample).
# In a real scenario, this would be part of a feature engineering pipeline.

df_with_label_sample = con.sql(f"""
    WITH daily_performance AS (
        SELECT
            content_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            LEAD(gsc_clicks, 1) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS gsc_clicks_next_day
        FROM
            read_parquet('{table_path}')
    )
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_clicks_next_day
    FROM
        daily_performance
    WHERE
        gsc_clicks_next_day IS NOT NULL -- Exclude rows where next day's clicks are not available
    ORDER BY content_hash_id, report_date
    LIMIT 10
""").fetchdf()

print(f"\nSample with simulated 'gsc_clicks_next_day' label:")
display(df_with_label_sample)


Verifying fields for month: 2026-03
Sample of identified fields from 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet':


,content_hash_id,report_date,gsc_impressions,gsc_clicks
0,content_000005d4ced12088,2026-03-01,0,0
1,content_000005d4ced12088,2026-03-02,0,0
2,content_000005d4ced12088,2026-03-03,1,0
3,content_000005d4ced12088,2026-03-04,4,0
4,content_000005d4ced12088,2026-03-05,2,0
5,content_000005d4ced12088,2026-03-06,3,0
6,content_000005d4ced12088,2026-03-07,0,0
7,content_000005d4ced12088,2026-03-08,0,0
8,content_000005d4ced12088,2026-03-09,0,0
9,content_000005d4ced12088,2026-03-10,1,0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Sample with simulated 'gsc_clicks_next_day' label:


,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_clicks_next_day
0,content_000005d4ced12088,2026-03-01,0,0,0
1,content_000005d4ced12088,2026-03-02,0,0,0
2,content_000005d4ced12088,2026-03-03,1,0,0
3,content_000005d4ced12088,2026-03-04,4,0,0
4,content_000005d4ced12088,2026-03-05,2,0,0
5,content_000005d4ced12088,2026-03-06,3,0,0
6,content_000005d4ced12088,2026-03-07,0,0,0
7,content_000005d4ced12088,2026-03-08,0,0,0
8,content_000005d4ced12088,2026-03-09,0,0,0
9,content_000005d4ced12088,2026-03-10,1,0,0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
target_month = '2026-03'
table_path = f"{rel}/fact_content_daily_performance/month={target_month}/*.parquet"

print(f"Verifying data availability for month: {target_month}")

# Availability check: Filter for rows where key metrics are not null
# and show the count before and after filtering.

# Original row count (already checked, but for context here):
original_count = con.sql(f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{table_path}')
""").fetchdf()

print(f"Total rows in {target_month}: {original_count['total_rows'].iloc[0]}")

# Filter for rows where impressions and clicks are not null/valid
# The problem description specifically mentioned 'filter with IS TRUE', which implies
# checking for boolean conditions or non-null values for key columns.

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS available_rows
    FROM
        read_parquet('{table_path}')
    WHERE
        gsc_impressions IS NOT NULL         -- Corrected from impressions
        AND gsc_clicks IS NOT NULL          -- Corrected from clicks
        AND gsc_impressions >= 0 AND gsc_clicks >= 0 -- Assuming non-negative values are valid
""").fetchdf()

print(f"\nAvailable rows after filtering (gsc_impressions, gsc_clicks IS NOT NULL and >= 0):")
display(availability_check)

# Calculate percentage of available rows
percentage_available = (availability_check['available_rows'].iloc[0] / original_count['total_rows'].iloc[0]) * 100
print(f"Percentage of available rows: {percentage_available:.2f}%")

Verifying data availability for month: 2026-03
Total rows in 2026-03: 9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Available rows after filtering (gsc_impressions, gsc_clicks IS NOT NULL and >= 0):


,available_rows
0,9841378


Percentage of available rows: 100.00%


## 5. Build Five Features

*Build a small feature frame, with an explanation for each: "knowable at the decision moment because…"*

Based on our data contract, we'll construct a feature set that includes direct metrics and one derived feature. Each feature is selected because it would be available at the time a prediction is made for `gsc_clicks_next_day`.

**Features to be included:**

1.  **`gsc_impressions`**: The number of Google Search Console impressions for a content item on a given `report_date`.
    *Knowable at the decision moment because:* This is a historical metric, directly observed on the `report_date` and available before the `gsc_clicks_next_day` would occur.

2.  **`gsc_clicks`**: The number of Google Search Console clicks for a content item on a given `report_date`.
    *Knowable at the decision moment because:* Similar to impressions, this is a historical metric, observed on the `report_date` and available prior to the label's observation.

3.  **`gsc_ctr_7d_ma`**: The 7-day moving average of Click-Through Rate (CTR) for the content item, ending on the `report_date`.
    *Knowable at the decision moment because:* This is a derived feature calculated from historical `gsc_clicks` and `gsc_impressions` over the past 7 days (including the `report_date`). All data points used in its calculation are available before the `gsc_clicks_next_day` would be observed. It provides a smoothed trend of content performance, reducing noise from daily fluctuations.

In [16]:
target_month = '2026-03'
table_path = f"{rel}/fact_content_daily_performance/month={target_month}/*.parquet"

print(f"Building feature frame for month: {target_month}")

# Construct the feature set including the derived gsc_ctr_7d_ma and the label
features_df = con.sql(f"""
    WITH daily_data AS (
        SELECT
            content_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            -- Calculate raw CTR, handling division by zero
            COALESCE(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0), 0) AS gsc_ctr,
            -- Calculate gsc_clicks_next_day as the label
            LEAD(gsc_clicks, 1) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS gsc_clicks_next_day
        FROM
            read_parquet('{table_path}')
    )
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        -- Calculate 7-day moving average of gsc_ctr
        AVG(gsc_ctr) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS gsc_ctr_7d_ma,
        gsc_clicks_next_day
    FROM
        daily_data
    WHERE
        gsc_clicks_next_day IS NOT NULL -- Only include rows where the label is available
    ORDER BY content_hash_id, report_date
    LIMIT 20 -- Show a small sample
""").fetchdf()

print("\nSample of the constructed feature frame with label:")
display(features_df)


Building feature frame for month: 2026-03


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Sample of the constructed feature frame with label:


,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_ctr_7d_ma,gsc_clicks_next_day
0,content_000005d4ced12088,2026-03-01,0,0,0.0,0
1,content_000005d4ced12088,2026-03-02,0,0,0.0,0
2,content_000005d4ced12088,2026-03-03,1,0,0.0,0
3,content_000005d4ced12088,2026-03-04,4,0,0.0,0
4,content_000005d4ced12088,2026-03-05,2,0,0.0,0
5,content_000005d4ced12088,2026-03-06,3,0,0.0,0
6,content_000005d4ced12088,2026-03-07,0,0,0.0,0
7,content_000005d4ced12088,2026-03-08,0,0,0.0,0
8,content_000005d4ced12088,2026-03-09,0,0,0.0,0
9,content_000005d4ced12088,2026-03-10,1,0,0.0,0


## 6. The Trap: Demonstrating Data Leakage

*Add one label-derived column (leakage), observe score jump, then delete it and keep the honest number.*

Data leakage occurs when information that would not be available at prediction time is used to train a model. To illustrate this, we will intentionally create a leaky feature: `gsc_clicks_on_next_day_leaky_feature`. This feature directly uses the `gsc_clicks` from the *next day*, which is precisely what our label `gsc_clicks_next_day` represents.

In a real-world scenario, a model trained with such a feature would show excellent performance during training and validation because it's essentially seeing the answer. However, it would perform poorly in production, as `gsc_clicks` for the next day would not be known when making a prediction.

We will add this leaky feature, sample the data, and mentally note that if we were to build a model, its performance would be misleadingly high. Then, we will verbally confirm its removal from the final feature set.

In [17]:
target_month = '2026-03'
table_path = f"{rel}/fact_content_daily_performance/month={target_month}/*.parquet"

print(f"Demonstrating data leakage for month: {target_month}")

# Construct the feature set with the intentionally leaky feature
leaky_features_df = con.sql(f"""
    WITH daily_data AS (
        SELECT
            content_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            -- Calculate raw CTR, handling division by zero
            COALESCE(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0), 0) AS gsc_ctr,
            -- Calculate gsc_clicks_next_day as the true label
            LEAD(gsc_clicks, 1) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS gsc_clicks_next_day,
            -- *** LEAKY FEATURE: This is the 'trap' ***
            -- It directly uses gsc_clicks from the next day, which is the label itself.
            LEAD(gsc_clicks, 1) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS gsc_clicks_on_next_day_leaky_feature
        FROM
            read_parquet('{table_path}')
    )
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        AVG(gsc_ctr) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS gsc_ctr_7d_ma,
        gsc_clicks_on_next_day_leaky_feature, -- Include the leaky feature
        gsc_clicks_next_day
    FROM
        daily_data
    WHERE
        gsc_clicks_next_day IS NOT NULL -- Only include rows where the label is available
    ORDER BY content_hash_id, report_date
    LIMIT 20 -- Show a small sample
""").fetchdf()

print("\nSample of feature frame with the leaky feature 'gsc_clicks_on_next_day_leaky_feature':")
display(leaky_features_df)

print("\n**Explanation of the Trap:**\n")
print("Notice how 'gsc_clicks_on_next_day_leaky_feature' is identical to 'gsc_clicks_next_day'.")
print("If a model were trained with this feature, it would have perfect or near-perfect predictive power on historical data.")
print("However, in a real production environment, 'gsc_clicks_on_next_day_leaky_feature' would not be available when predicting 'gsc_clicks_next_day'.")
print("Therefore, this feature *must* be excluded from the final feature set to prevent data leakage and ensure an honest evaluation of model performance.")


Demonstrating data leakage for month: 2026-03


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Sample of feature frame with the leaky feature 'gsc_clicks_on_next_day_leaky_feature':


,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_ctr_7d_ma,gsc_clicks_on_next_day_leaky_feature,gsc_clicks_next_day
0,content_000005d4ced12088,2026-03-01,0,0,0.0,0,0
1,content_000005d4ced12088,2026-03-02,0,0,0.0,0,0
2,content_000005d4ced12088,2026-03-03,1,0,0.0,0,0
3,content_000005d4ced12088,2026-03-04,4,0,0.0,0,0
4,content_000005d4ced12088,2026-03-05,2,0,0.0,0,0
5,content_000005d4ced12088,2026-03-06,3,0,0.0,0,0
6,content_000005d4ced12088,2026-03-07,0,0,0.0,0,0
7,content_000005d4ced12088,2026-03-08,0,0,0.0,0,0
8,content_000005d4ced12088,2026-03-09,0,0,0.0,0,0
9,content_000005d4ced12088,2026-03-10,1,0,0.0,0,0



**Explanation of the Trap:**

Notice how 'gsc_clicks_on_next_day_leaky_feature' is identical to 'gsc_clicks_next_day'.
If a model were trained with this feature, it would have perfect or near-perfect predictive power on historical data.
However, in a real production environment, 'gsc_clicks_on_next_day_leaky_feature' would not be available when predicting 'gsc_clicks_next_day'.
Therefore, this feature *must* be excluded from the final feature set to prevent data leakage and ensure an honest evaluation of model performance.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One named limitation of this data slice (March 2026) is the **limited historical context for new content items or recent changes**. For any `content_hash_id` that is new or has undergone significant changes just before or during March 2026, the 7-day moving average of CTR (`gsc_ctr_7d_ma`) will not be fully informed by a complete 7 days of historical data at the beginning of the month. This can lead to less reliable feature values for such items during their initial period in the observed window.

This means that models trained on this slice might struggle to accurately predict performance for very new content or content with recent, abrupt changes, as their early historical trends are incomplete or not fully representative. We are essentially assuming a certain level of history for all content, which might not always be true, particularly if the content was first published on March 1st, 2026, or later.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.